# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계3 : 응급상황 연계(추천)**

## **0.미션**

단계 3에서는, 응급상황의 음성을 인식해서 텍스트로 변환하고, 변환된 텍스트를 다시 요약 및 핵심키워드 도출 작업을 수행합니다.  
이를 위해 사전학습된 모델을 API로 연결하여 활용합니다.

### 미션4 : 응급실 추천
* 응급실 위치와 응급전화 발신자 위치 기반 추천
* 두 좌표간 직선거리(Haversine)
    * 1) 500여 곳 응급실에 대해서, 거리 기반 가까운 응급실 찾기
    * 2) 좌표 구간을 설정하여 대상 응급실 범위를 좁힌 후, 거리 기반 가까운 응급실 찾기


## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project6_2)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/mini-project-6-1/' ## 수정 필요

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

In [3]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r /content/drive/MyDrive/mini-project-6-1/requirements.txt ## 경로 수정 필요

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


#### 2) 라이브러리 로딩

In [43]:
#필요한 라이브러리 설치 및 불러우기
import os
import pandas as pd
import numpy as np

from haversine import haversine
import requests
import json

# 더 필요한 라이브러리 추가 -------------

import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import openai
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from geopy.distance import geodesic
from geopy import Point
from haversine import haversine,haversine_vector, Unit
from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

### (3) 데이터 로딩
* 단계1에서 수집한 응급실 정보를 불러와서 데이터프레임으로 저장합니다.

In [5]:
situation = pd.read_csv(path+'ktas.csv', index_col=0)

situation

,text,label
0,환자가 심각한 호흡곤란으로 숨을 쉬기 어려워하며 피부가 푸르스름하게 변색된 상태 ...,1
1,환자가 얼굴이 창백하고 식은땀을 흘리며 맥박이 거의 잡히지 않는 쇼크 상태 ...,1
2,"환자가 바닥에 쓰러져 의식을 잃었고, 주변의 소리에 반응하지 않으며, 몸이 움직이지...",1
3,"심장마비로 인해 환자가 갑작스럽게 가슴을 부여잡고 쓰러지며, 호흡이 중단된 상태 ...",1
4,음주와 무관하게 갑자기 의식을 잃고 눈을 뜨지 못하며 호흡이 어려운 상태 ...,1
...,...,...
495,환자가 기침과 함께 가벼운 두통을 겪는 상태,4
496,환자가 만성적인 두통과 함께 체온 상승을 보이는 상태,4
497,환자가 경미한 발열을 동반하는 통증을 호소하는 상태,4
498,환자가 약간의 구토를 보이며 복통을 겪고 있는 상태,4


In [29]:
emergency = pd.read_csv(path+'응급실 정보.csv')

emergency

,병원이름,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도
0,(의)내경의료재단울산제일병원,울산광역시 남구 남산로354번길 26 (신정동),응급실운영신고기관,052-220-3300,052-220-3334,35.548238,129.307011
1,(의)서일의료재단기장병원,부산광역시 기장군 기장읍 대청로72번길 6,지역응급의료기관,051-723-0171,051-723-2119,35.236029,129.216492
2,(의)성세의료재단 뉴성민병원,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",지역응급의료기관,032-726-1000,032-726-1190,37.508994,126.669479
3,(의)영문의료재단다보스병원,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",지역응급의료센터,031-8021-2114,031-8021-2130,37.234641,127.210499
4,(의)효심의료재단용인서울병원,경기도 용인시 처인구 고림로 81 (고림동),지역응급의료기관,031-337-0114,031-336-0119,37.240316,127.214491
...,...,...,...,...,...,...,...
520,효산의료재단안양샘병원,"경기도 안양시 만안구 삼덕로 9 (안양동, 안양샘병원)",지역응급의료센터,031-467-9717,031-467-9119,37.393404,126.924477
521,효산의료재단지샘병원,"경기도 군포시 군포로 591 (당동, (G샘병원)군포샘병원)",지역응급의료센터,031-389-3000,031-389-3119,37.358645,126.947360
522,효성시티병원,부산광역시 해운대구 해운대로 135 (재송동),응급실운영신고기관,051-709-3000,051-709-3119,35.185413,129.121459
523,흑룡의원,인천광역시 옹진군 백령면 백령로 831,응급실운영신고기관,032-837-6873,032-837-3153,37.959524,124.665499


In [7]:
emergency['응급의료기관 종류'].value_counts()

,count
응급의료기관 종류,
지역응급의료기관,231
지역응급의료센터,136
응급실운영신고기관,114
권역응급의료센터,44


In [8]:
emergency.isna().sum()

,0
병원이름,0
주소,0
응급의료기관 종류,0
전화번호 1,0
전화번호 3,15
위도,0
경도,0


## **2. 응급실 추천**


### (1) 직선거리 계산
- haversine formula
    * Haversine은 두 지점 간의 거리를 구할 때 사용하는 수학 공식으로, 지구의 구형 구조를 고려하여 위도와 경도를 기반으로 직선 거리를 계산한다.
- 세부사항
    * 하버사인 함수를 활용


#### 1) 하버사인 함수 사용 연습
* 임의의 두 좌표간 거리 계산
    * 응급실 데이터프레임을 열어서
    * 응급실 두 곳의 좌표를 확인하고
    * 두 지점의 거리를 계산해 봅시다
* 사용법 : haversine((위도1, 경도1), (위도2, 경도2), unit='km')


In [9]:
haversine((36.72535, 127.263425), (37.524212, 128.092342), unit='km')

115.2877182176334

#### 2) 가장 가까운 응급실 3곳 추천하기1
* 세부사항
    * 입력된 좌표와 전체 응급실과 거리를 계산한 후
    * 가장 가까운 거리의 응급실 3 곳을 반환합니다.
* 이를 함수로 생성하고 테스트 해 봅시다.

In [10]:
point = (36.5035440021842,127.252940982958)

In [11]:
def recommend_hospital1(point, df):
  df['거리'] = df.apply(lambda row: haversine(point, (row['위도'], row['경도']), unit='km'), axis=1)
  df_sorted = df.sort_values(by='거리')

  return df_sorted[:3]

In [12]:
hospitals1 = recommend_hospital1(point, emergency)

hospitals1[:3]

,병원이름,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,distance
199,세종충남대학교병원,"세종특별자치시 보듬7로 20, 세종충남대학교병원 (도담동)",지역응급의료센터,1800-3114,044-995-3010,36.519513,127.257943,1.831114
274,의료법인 영제 의료재단 엔케이세종병원,"세종특별자치시 한누리대로 161, 세종시 NK리움힐 상가 1~8층 (나성동)",지역응급의료기관,044-850-7700,044-850-7705,36.479623,127.262256,2.787254
457,충청남도공주의료원,충청남도 공주시 무령로 77 (웅진동),지역응급의료기관,041-962-1111,041-962-1365,36.459055,127.109216,13.769346


#### 3) 가장 가까운 응급실 3곳 추천하기2
* 문제점 : 입력 받은 좌표와 응급실 전체와의 거리를 모두 계산하는 것은 비효율 적입니다.
* 해결 방안 : 그래서 입력 받은 좌표를 기준으로 일정 범위 내에 해당되는 응급실에 대해서 거리를 계산하고 추천하도록 기존 함수를 수정 합니다.
* hint :
    * 입력 받은 위도, 경도 값에 ± α 하여 일정 범위 구간을 정하고
    * 응급실 정보에서 해당 구간을 먼저 조회한 후
    * 거리 계산

In [13]:
def recommend_hospital2(point, df, range=0.1):
  lat_min = point[0] - range
  lat_max = point[0] + range
  lon_min = point[1] - range
  lon_max = point[1] + range

  df_cal = df.loc[(df['위도'].between(lat_min, lat_max)) & (df['경도'].between(lon_min, lon_max))]
  df_cal['거리'] = df_cal.apply(lambda row: haversine(point, (row['위도'], row['경도']), unit='km'), axis=1)
  df_sorted = df_cal.sort_values(by='거리')

  return df_sorted[:3]

In [14]:
hospitals2 = recommend_hospital2(point, emergency)

hospitals2[:3]

,병원이름,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,distance,거리
199,세종충남대학교병원,"세종특별자치시 보듬7로 20, 세종충남대학교병원 (도담동)",지역응급의료센터,1800-3114,044-995-3010,36.519513,127.257943,1.831114,1.831114
274,의료법인 영제 의료재단 엔케이세종병원,"세종특별자치시 한누리대로 161, 세종시 NK리움힐 상가 1~8층 (나성동)",지역응급의료기관,044-850-7700,044-850-7705,36.479623,127.262256,2.787254,2.787254


### 1번 과정 파이프라인

In [30]:
def load_file(filepath):
    with open(filepath, 'r') as file:
        return file.readline().strip()

# API 키 로드 및 환경변수 설정
openai.api_key = load_file(path + 'api_key.txt')
os.environ['OPENAI_API_KEY'] = openai.api_key

In [31]:
location = pd.read_excel(path+'audio_location.xlsx')

location = location.loc[:, ~location.columns.str.contains('^Unnamed')]
location = location.dropna(axis=1, how='all')
location = location.dropna(axis=0, how='all')

location

,filename,위도,경도
0,audio1.mp3,37.358618,127.115036
1,audio2.mp3,36.815571,127.128844
2,audio3.mp3,37.538435,126.989828
3,audio4.mp3,35.185745,129.076541
4,audio5.mp3,36.503544,127.252941
5,1-1.m4a,37.358618,127.115036
6,1-2.m4a,36.815571,127.128844
7,1-3.m4a,37.538435,126.989828
8,2-1.m4a,35.185745,129.076541
9,2-2.m4a,36.503544,127.252941


In [32]:
# location = pd.read_csv('/content/drive/MyDrive/mini-project-6-1/latitude_longitude.csv', encoding='euc-kr')

# location

In [33]:
audio_path = path + 'audio/'

# 음성파일 이름을 리스트에 담기
file_names = [f for f in os.listdir(audio_path) if os.path.isfile(os.path.join(audio_path, f))]
print(file_names)

['audio2.mp3', 'audio5.mp3', 'audio4.mp3', 'audio3.mp3', 'audio1.mp3']


In [34]:
audio_path2 = path + 'self_audio/'

In [35]:
def audio_summary(audio_path, filename, location):
  latitude = location['위도'].loc[location['filename'] == filename].iloc[0]
  longitude = location['경도'].loc[location['filename'] == filename].iloc[0]

  # OpenAI 클라이언트 생성
  client = OpenAI()

  audio_file = open(audio_path + filename, "rb")
  transcript = client.audio.transcriptions.create(
      file=audio_file,
      model="whisper-1",
      language="ko",
      response_format="text",
  )

  # 시스템 역할과 응답 형식 지정
  system_role = '''당신은 응급상황이 발생했을 때, 119 신고자의 전화 통화 내용을 요약하는 어시스턴트입니다.
  통화 스크립트를 듣고 내용을 요약해주세요.
  응답은 다음의 형식을 지켜주세요
  {~~한 상태}
  '''

  # 입력데이터를 GPT-3.5-turbo에 전달하고 답변 받아오기
  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[
          {
              "role": "system",
              "content": system_role
          },
          {
              "role": "user",
              "content": transcript
          }
      ]
  )

  # 응답 받기
  answer = response.choices[0].message.content
  print(transcript, answer)

  return answer, latitude, longitude

In [36]:
audio_summary(audio_path2, '5-2.m4a', location)

지금 탐방동입니다. 친구가 장염 증상으로 복통과 설사를 겪고 있는데 상태가 심각하진 않은데 진료가 필요해 보입니다. 응급상황은 아니고 가까운 병원으로 갈 수 있도록 구급차를 요청드립니다.
 친구가 장염 증상으로 복통과 설사를 겪고 있지만 응급상황은 아니고 상태가 심각하지 않은 상태입니다. 구급차를 요청하여 가까운 병원으로 이동할 필요가 있습니다.


('친구가 장염 증상으로 복통과 설사를 겪고 있지만 응급상황은 아니고 상태가 심각하지 않은 상태입니다. 구급차를 요청하여 가까운 병원으로 이동할 필요가 있습니다.',
 35.1857454699834,
 129.07654056555)

### 2번 과정 파이프라인

In [37]:
save_directory = path+'fine_tuned_bert/2/'

# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(save_directory)

In [38]:
# 데이터 예측 함수
def classify_situation(audio_path, filename, location, model, tokenizer):
  text, latitude, longitude = audio_summary(audio_path, filename, location)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  # 입력 문장 토크나이징
  inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
  inputs = {key: value.to(device) for key, value in inputs.items()}  # 각 텐서를 GPU로 이동

  # 모델 예측
  with torch.no_grad():
      outputs = model(**inputs)

  # 로짓을 소프트맥스로 변환하여 확률 계산
  logits = outputs.logits
  probabilities = logits.softmax(dim=1)

  # 가장 높은 확률을 가진 클래스 선택
  pred = torch.argmax(probabilities, dim=-1).item() + 1
  print(pred, '등급')

  return pred, latitude, longitude

In [39]:
classify_situation(audio_path2, '5-2.m4a', location, model, tokenizer)

지금 탐방동입니다. 친구가 장염 증상으로 복통과 설사를 겪고 있는데 상태가 심각하진 않은데 진료가 필요해 보입니다. 응급상황은 아니고 가까운 병원으로 갈 수 있도록 구급차를 요청드립니다.
 {친구가 장염 증상으로 복통과 설사를 겪고 있는 상황, 응급상황은 아니지만 진료가 필요하다고 판단하여 구급차를 요청}
3 등급


(3, 35.1857454699834, 129.07654056555)

### 3번 과정 파이프라인

In [40]:
# 데이터 예측 함수
def recommend_hospital(audio_path, filename, location, model, tokenizer, hospitals, range=0.1):
  pred, latitude, longitude = classify_situation(audio_path, filename, location, model, tokenizer)

  if pred > 3:
    return "가까운 병원을 이용하세요", pred

  lat_min = latitude - range
  lat_max = latitude + range
  lon_min = longitude - range
  lon_max = longitude + range

  df_cal = hospitals.loc[(hospitals['위도'].between(lat_min, lat_max)) & (hospitals['경도'].between(lon_min, lon_max))]
  df_cal['거리'] = df_cal.apply(lambda row: haversine((latitude, longitude), (row['위도'], row['경도']), unit='km'), axis=1)
  df_sorted = df_cal.sort_values(by='거리')

  return df_sorted[:3], pred

In [41]:
hospitals3, pred = recommend_hospital(audio_path2, '4-2.m4a', location, model, tokenizer, emergency)

hospitals3

지금 제 아내가 심한 통증을 호소하고 있습니다. 고요일에 동반되고 약을 먹어도 상태가 나아지지 않고 통증이 갈수록 심해지고 있습니다. 응급실 가야 될 것 같습니다. 가능하면 구급차를 빨리 보내주시면 감사하겠습니다.
 {아내가 심한 통증을 호소하고 있으며, 통증은 악화되는 중이고 약물 복용으로 나아지지 않음. 응급실 방문이 필요하며, 구급차 신속히 요청하고 있음.}
3 등급


,병원이름,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,거리
156,분당서울대학교병원,"경기도 성남시 분당구 구미로173번길 82 (구미동, 분당서울대학교병원)",권역응급의료센터,031-787-2114,031-787-3119,37.352026,127.124484,1.111162
116,대진의료재단분당제생병원,경기도 성남시 분당구 서현로180번길 20 (서현동),지역응급의료센터,031-779-0114,031-779-0119,37.387871,127.121328,3.299964
76,국군수도병원,경기도 성남시 분당구 새마을로177번길 81 (율동),지역응급의료기관,031-725-6075,031-725-6119,37.391867,127.148586,4.738899


## 전체 과정 클래스 RecommendHospital3

In [64]:
naver_id,naver_key='',''
loca = (36.5035440021842,127.252940982958)

In [78]:
class RecommendHospital3:
    def __init__(self, filename, naver_id, naver_key, path):
        self.filename = filename
        self.th = 3
        self.id = naver_id
        self.key = naver_key
        self.path = path
        self.data = pd.read_csv(self.path + '응급실 정보.csv', encoding='utf-8') ## 수정 필요
        self.audio_path = path + 'self_audio/' ## 수정 필요

    def load_api_key(filepath):
      with open(filepath, 'r') as file:
          return file.readline().strip()
      openai.api_key = load_file(path + 'api_key.txt')
      os.environ['OPENAI_API_KEY'] = openai.api_key

    def audio_summary(self):
      location = pd.read_excel(self.path + 'audio_location.xlsx')
      latitude = location['위도'].loc[location['filename'] == self.filename].iloc[0]
      longitude = location['경도'].loc[location['filename'] == self.filename].iloc[0]

      # OpenAI 클라이언트 생성
      client = OpenAI()

      audio_file = open(self.audio_path + self.filename, "rb")
      transcript = client.audio.transcriptions.create(
          file=audio_file,
          model="whisper-1",
          language="ko",
          response_format="text",
      )

      # 시스템 역할과 응답 형식 지정
      system_role = '''당신은 의사입니다.
      당신은 응급상황이 발생했을 때, 119 신고자의 전화 통화 내용을 듣고, 더 실력있는 의사에게 응급상황인지 아닌지 구별할 수 있게끔 문장을 가다듬는 업무를 하고 있습니다.
      응답은 다음의 형식을 지켜주세요.
      응급상황인지 아닌지를 구별할 수 있게끔 문장을 제공해야 합니다. (ex) 숨을 쉬지 않음, 경미한 발열, 가벼운 탈수 증세 등)
      신고자는 정확한 정보를 제공할 수도, 아닐 수도 있습니다.(정말 긴급한 상황에서의 구급차 요청, 장난전화로 구급차 요청)
      구급차를 제공하는 기준은 한국 응급환자 중증도 분류기준에 따라 1,2,3등급에 해당하면 구급차를 제공합니다.
      구급차를 제공하지 않는 기준은 한국 응급환자 중증도 분류기준에 따라 4, 5등급에 해당하면 구급차를 제공하지 않습니다.
      문장에는 다음의 정보를 포함할 수 있습니다. (ex) 증상, 증세, 각종 지병, 신고자의 주관적인 판단(ex)신고자는 위급상황이라고 판단하고 있습니다.), 상황을 파악할 수 있는 객관적인 사실 등)
      문장에는 다음의 정보를 포함할 수 없습니다. (ex) 신고자의 위치)

      (예시) Python 3
      text = '손을 살짝 베었는데, 구급차 불러주세요.'
      test = text_summary(text)
      print(test)
      => {"상황": \"환자는 손을 살짝 베였습니다.\"}
      => {"환자의 요구, 질문 및 판단": \"구급차를 요청하고 있습니다. \"}
      => {"1차 판단": \"손을 살짝 베었기 때문에, 간단한 조치만 취하면 될 것 같습니다. \"}

      이 때 1차 판단에 해당되는 내용만 출력해주세요.
      '''

      # 입력데이터를 GPT-3.5-turbo에 전달하고 답변 받아오기
      response = client.chat.completions.create(
          model="gpt-3.5-turbo",
          messages=[
              {
                  "role": "system",
                  "content": system_role
              },
              {
                  "role": "user",
                  "content": transcript
              }
          ]
      )

      # 응답 받기
      answer = response.choices[0].message.content
      print(transcript, answer)

      return answer, latitude, longitude

    # 데이터 예측 함수
    def classify_situation(self):
      text, latitude, longitude = self.audio_summary()

      save_directory = self.path + 'fine_tuned_bert/2/' ## 수정 필요

      # 모델 로드
      model = AutoModelForSequenceClassification.from_pretrained(save_directory)

      # 토크나이저 로드
      tokenizer = AutoTokenizer.from_pretrained(save_directory)

      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

      # 입력 문장 토크나이징
      inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
      inputs = {key: value.to(device) for key, value in inputs.items()}  # 각 텐서를 GPU로 이동

      # 모델 예측
      with torch.no_grad():
          outputs = model(**inputs)

      # 로짓을 소프트맥스로 변환하여 확률 계산
      logits = outputs.logits
      probabilities = logits.softmax(dim=1)

      # 가장 높은 확률을 가진 클래스 선택
      pred = torch.argmax(probabilities, dim=-1).item() + 1
      print(pred, '등급')

      return pred, latitude, longitude

    def km_to_lat_lon(self, km_north, km_east, latitude=37.5665, longitude=126.978):
        start_point = Point(latitude, longitude)
        north_point = geodesic(kilometers=km_north).destination(start_point, bearing=0)
        east_point = geodesic(kilometers=km_east).destination(north_point, bearing=90)
        return east_point.latitude - latitude, east_point.longitude - longitude

    def get_dist(self, start_lat, start_lng, dest_lat, dest_lng):
        url = "https://naveropenapi.apigw.ntruss.com/map-direction/v1/driving"
        headers = {
            "X-NCP-APIGW-API-KEY-ID": self.id,
            "X-NCP-APIGW-API-KEY": self.key,
        }
        params = {
            "start": f"{start_lng},{start_lat}",
            "goal": f"{dest_lng},{dest_lat}",
            "option": "trafast"
        }

        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            response_data = response.json()
            try:
                return response_data['route']['trafast'][0]['summary']
            except KeyError:
                return None
        else:
            return None

    def convert_milliseconds(self, milliseconds):
        hours = milliseconds // (1000 * 60 * 60)
        minutes = (milliseconds // (1000 * 60)) % 60
        return hours, minutes

    def recommend_hospital(self):
        pred, lat, lon = self.classify_situation()

        if pred > 3:
          return None, lat, lon

        x, y = self.km_to_lat_lon(self.th, self.th, lat, lon)

        filter_lst = self.data.loc[
            ((lat - x < self.data['위도']) &
             (self.data['위도'] < lat + x)) &
            ((lon - y < self.data['경도']) &
             (self.data['경도'] < lon + y))
        ].copy()

        while len(filter_lst) < 5:
            self.th += 5
            x, y = self.km_to_lat_lon(self.th, self.th, lat, lon)
            filter_lst = self.data.loc[
                ((lat - x < self.data['위도']) &
                 (self.data['위도'] < lat + x)) &
                ((lon - y < self.data['경도']) &
                 (self.data['경도'] < lon + y))
            ].copy()

            if len(filter_lst) >= 5:
                break

        if filter_lst.empty:
            return None

        locations = list(zip(filter_lst['위도'], filter_lst['경도']))
        filter_lst['distance'] = haversine_vector(
            locations,
            np.tile([lat, lon], (len(filter_lst), 1)),
            unit=Unit.KILOMETERS
        )
        return filter_lst.sort_values(by='distance').head(5), lat, lon

    def search_map(self):
        filter_lst, lat, lon = self.recommend_hospital()
        if filter_lst is None:
            return "가까운 병원을 찾아가는 것을 추천드립니다."

        total_result = {'목적지': [], '경과시간': [], '거리': []}

        for i in range(len(filter_lst)):
            hospital = filter_lst.iloc[i]
            result = self.get_dist(
                lat, lon,
                hospital['위도'], hospital['경도']
            )
            if result:
                total_result['목적지'].append(hospital['병원이름'])
                hours, minutes = self.convert_milliseconds(result['duration'])
                total_result['경과시간'].append(f"{hours}시간 {minutes}분")
                total_result['거리'].append(result['distance'])

        return total_result


In [79]:
RecommendHospital3('5-1.m4a', naver_id, naver_key, path).search_map()

지금 편방동에 있는 집입니다. 며칠 전부터 감기 증상을 보이던 가족이 지금도 가벼운 기침과 콧물을 흘리고 있습니다. 응급상황은 아니지만 약국이 문을 닫았고 병원 진료를 받고 싶습니다. 구급차 요청드리지만 급하지 않으니 천천히 오셔도 됩니다.
 가족이 가벼운 기침과 콧물 증상을 보이고 있으며 응급상황은 아니라고 판단됩니다. 구급차 요청은 급하지 않으며, 병원 진료를 받고 싶다는 신고자의 의견이 있습니다.
4 등급


'가까운 병원을 찾아가는 것을 추천드립니다.'

### (2) [조 과제]고도화 : naver 지도 api 사용

* 이 부분은 조별 과제로 수행하게 됩니다.(개인과제 아님!)

* 세부사항
    * 두 지점간, 최단 도로거리, 소요 시간을 계산하는 함수를 생성하시오.
    * 함수 내용
        * 입력 : 두 지점의 위도, 경도, 네이버클라우드id, 암호키
        * 출력 : 도로거리(km)
    
    * 네이버 Maps API 활용
        * 사용할 API : Direction 5
        * 가이드 : https://guide.ncloud-docs.com/docs/ko/maps-direction5-api
        * 가이드를 활용해서 url, header, params를 구성합니다.
        * params의 옵션은 'trafast' (실시간 빠른 길 옵션)을 선택하시오.

#### 1) maps 클라이언트ID, 키 로딩

#### 2) 함수 생성

In [ ]:
def get_dist(start_lat, start_lng, dest_lat, dest_lng):
    url = "https://naveropenapi.apigw.ntruss.com/map-direction/v1/driving"
    headers = {
        "X-NCP-APIGW-API-KEY-ID": '',
        "X-NCP-APIGW-API-KEY": '',
    }
    params = {
        "start": f"{start_lng},{start_lat}",  # 출발지 (경도, 위도)
        "goal": f"{dest_lng},{dest_lat}",    # 목적지 (경도, 위도)
        "option": "trafast"  # 실시간 빠른 길 옵션
    }

    # 요청하고, 답변 받아오기
    response =requests.get(url, params = params,headers=headers)
    response_data = response.json()

    dist = response_data['route']['trafast'][0]['summary']['distance']  # m(미터)
    duration = dist = response_data['route']['trafast'][0]['summary']['duration']
    dist = dist/1000
    hours = duration // 3600000  # 1시간 = 3600000 밀리초
    minutes = (duration % 3600000) // 60000  # 1분 = 60000 밀리초
    return dist,hours,minutes

* 테스트

In [ ]:
emergency

#### 3) 응급실 추천
* recommend_hospital2 함수를 참조해서 recommend_hospital3 만들기
    * 거리 계산 부분을 get_dist 함수로 대체
    * 입력 부분 수정

In [ ]:
def recommend_hospital3(lat, lng):
    # 초기 반경 설정
    lat_offset = 0.2
    lng_offset = 0.2

    hospitals = []  # 병원 정보가 담긴 리스트
    naver = []  # 병원 정보가 담긴 리스트 (딕셔너리 형태)

    # 병원 정보를 dataframe에서 가져오기 (필요한 컬럼만)
    hospital_data = emergency[['병원이름', '위도', '경도']]

    # 병원 수가 5개 이상일 때까지 범위 내 병원 찾기
    while len(hospitals) < 5 :
        # 좌표 범위 계산
        lat_range = (lat - lat_offset, lat + lat_offset)
        lng_range = (lng - lng_offset, lng + lng_offset)

        mask = (hospital_data['위도'] >= lat_range[0]) & (hospital_data['위도'] <= lat_range[1]) & \
               (hospital_data['경도'] >= lng_range[0]) & (hospital_data['경도'] <= lng_range[1])
        bound = hospital_data[mask]

        if len(bound) >= 5:
            for _, item in bound.iterrows():
                km = haversine((lat, lng), (item['위도'], item['경도']), unit='km')
                hospitals.append({'hospital': item['병원이름'], 'lat': item['위도'], 'lng': item['경도'], 'km': km})
        else:
            lat_offset += 0.045  # 5km 증가
            lng_offset += 0.045  # 5km 증가

    sorted_bound = sorted(bound.to_dict(orient='records'), key=lambda x: haversine((lat, lng), (x['위도'], x['경도']), unit='km'))[:5]

    for item in sorted_bound:
        km, time, min_time = get_dist(lat, lng, item['위도'], item['경도'])

        naver.append({
            'hospital': item['병원이름'],
            'km': float(km),
            'time': time,
            'min': min_time
        })

    # naver 리스트에서 'km' 기준으로 가까운 병원 3개를 정렬하여 반환
    sorted_hospitals = sorted(naver, key=lambda x: x['km'])[:3]

    return sorted_hospitals

In [ ]:
result = recommend_hospital3(36.5035440021842,127.252940982958)
print(result)

## **Mission Complete!**

수고 많았습니다!